# Inbox Triage Agent --- Experimentation & Model Benchmark Notebook

**An ML-Powered System to Automatically Classify and Act on Your Emails**  
*Academic Machine Learning Project --- Email Data, Classification + Agent Layer*

---

### Pipeline Overview:
$$\text{Email Input} \longrightarrow \text{Pre-process} \longrightarrow \text{Feature Extraction} \longrightarrow \text{ML Classifiers (NB, LR, SVM)} \longrightarrow \text{Agent Action Layer}$$

### Supported Categories:
1. `URGENT`
2. `NEEDS-REPLY`
3. `FYI`
4. `SPAM`
5. `DELEGATE`

## 1. Setup & Environment Verification

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root in sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root configured: {PROJECT_ROOT}")

## 2. Exploratory Data Analysis (EDA) on Email Corpus
Loading the generated benchmark dataset reflecting Enron dynamics, SpamAssassin, and personal inbox workflows.

In [ ]:
data_path = os.path.join(PROJECT_ROOT, "data", "raw", "emails_raw.csv")
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} emails across columns: {list(df.columns)}")
display(df.head(5))

# Class distribution
plt.figure(figsize=(7, 3.5))
sns.countplot(data=df, x="label", palette="viridis", order=["URGENT", "NEEDS-REPLY", "FYI", "SPAM", "DELEGATE"])
plt.title("Class Distribution across Email Categories", fontweight="bold")
plt.xlabel("Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 3. Text Preprocessing Pipeline
Cleaning email subjects and bodies: lowercasing, HTML removal, tokenization, stopword filtering, and Porter stemming.

In [ ]:
from src.preprocessing import clean_text

sample_raw = df["body"].iloc[0]
sample_cleaned = clean_text(sample_raw)

print("--- Raw Text Sample ---")
print(sample_raw[:250])
print("\n--- Cleaned & Stemmed Text ---")
print(sample_cleaned[:250])

## 4. Hybrid Feature Extraction
Extracting:
- **Subject**: TF-IDF
- **Body**: Bag of Words (CountVectorizer)
- **Metadata**: Sender Domain, History Score, Hour of Day, Has Links, Has Attachments

In [ ]:
from src.features import EmailFeaturePipeline
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df["label"])

feature_pipe = EmailFeaturePipeline(max_subject_features=500, max_body_features=1500)
feature_pipe.fit(train_df)

X_train = feature_pipe.transform(train_df, feature_subset="all")
X_test = feature_pipe.transform(test_df, feature_subset="all")
y_train = train_df["label"].values
y_test = test_df["label"].values

print(f"Feature matrix shape: Train={X_train.shape}, Test={X_test.shape}")
print(f"Total hybrid feature dimensionality: {X_train.shape[1]}")

## 5. Model Training & Comparison
Evaluating the three core models against the presentation targets:
- Naive Bayes (PPT Ref: 78%)
- Logistic Regression (PPT Ref: 85%)
- SVM with RBF Kernel (PPT Ref: 88%)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from src.evaluation import evaluate_model_predictions, generate_comparison_table

models = {
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Logistic Regression": LogisticRegression(C=1.5, max_iter=1000, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", C=5.0, gamma="auto", probability=True, random_state=42)
}

results = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    metrics = evaluate_model_predictions(y_test, preds, model_name=name)
    results.append(metrics)

comparison_df = generate_comparison_table(results)
display(comparison_df)

## 6. Confusion Matrix Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels = ["URGENT", "NEEDS-REPLY", "FYI", "SPAM", "DELEGATE"]

for ax, res in zip(axes, results):
    cm = np.array(res["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f"{res['model_name']} Confusion Matrix", fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

plt.tight_layout()
plt.show()

## 7. Feature Ablation Study
Testing marginal contribution of Subject TF-IDF vs Body BoW vs Metadata.

In [ ]:
subsets = ["subject_only", "body_only", "text_only", "metadata_only", "all"]
ablation_results = []

for s in subsets:
    X_tr = feature_pipe.transform(train_df, feature_subset=s)
    X_te = feature_pipe.transform(test_df, feature_subset=s)
    clf = LogisticRegression(C=1.5, max_iter=1000, random_state=42)
    clf.fit(X_tr, y_train)
    acc = clf.score(X_te, y_test)
    ablation_results.append({"Feature Subset": s, "Feature Count": X_tr.shape[1], "Accuracy": acc})

display(pd.DataFrame(ablation_results))

## 8. Agent / Action Layer Demonstration
Verifying decoupled `Category -> Action` execution:

In [ ]:
from src.agent import AgentActionEngine

agent = AgentActionEngine()

demo_emails = [
    {"label": "URGENT", "email": {"subject": "Emergency: Database latency spike in prod", "sender": "ops@company.com"}},
    {"label": "NEEDS-REPLY", "email": {"subject": "Proposal review sync for next sprint", "sender": "alex@company.com"}},
    {"label": "FYI", "email": {"subject": "Weekly engineering digest newsletter", "sender": "news@company.com"}},
    {"label": "SPAM", "email": {"subject": "Congratulations you won free lottery prize", "sender": "winner@spam.xyz"}},
    {"label": "DELEGATE", "email": {"subject": "Customer ticket: Please delegate billing inquiry", "sender": "support@company.com"}}
]

for item in demo_emails:
    action = agent.execute_action(item["email"], category=item["label"])
    print(f"\nCategory: [{item['label']}]")
    print(f"Action:   {action['action_description']}")
    print(f"Details:  {action['details']}")
    if "suggested_reply" in action["simulated_execution"]:
        print("Suggested Draft Reply:\n" + action["simulated_execution"]["suggested_reply"])